In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

# Configuration du device (GPU si dispo)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"L'entraînement se fera sur : {device}")

# Hyperparamètres
BATCH_SIZE = 64
LR = 0.01
EPOCHS = 3  # On met peu d'époques pour l'exemple, augmentez à 10+ pour voir les résultats

L'entraînement se fera sur : cuda


Nous utilisons CIFAR-10, un dataset classique d'images 32x32 (Avion, Auto, Oiseau, Chat...).

In [2]:
# Transformations simples
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Chargement des données
trainset = torchvision.datasets.CIFAR10(root='data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=BATCH_SIZE,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=BATCH_SIZE,
                                         shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

Files already downloaded and verified


Files already downloaded and verified


Définition des Modèles (Teacher & Student)

In [7]:
class SimpleCNN(nn.Module):
    def __init__(self, num_channels=16):
        super(SimpleCNN, self).__init__()
        # Le paramètre num_channels permet de contrôler la taille du modèle
        self.conv1 = nn.Conv2d(3, num_channels, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(num_channels, num_channels*2, 3, padding=1)
        # Après 2 pooling sur 32x32, on a 8x8
        self.fc1 = nn.Linear(num_channels*2 * 8 * 8, 64)
        self.fc2 = nn.Linear(64, 10) # 10 classes

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, self.num_flat_features(x))
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

    def num_flat_features(self, x):
        size = x.size()[1:]
        num_features = 1
        for s in size:
            num_features *= s
        return num_features

# --- EXERCICE ---
# Créez un Teacher "Gros" (64 channels) et un Student "Petit" (16 channels)
teacher_model = SimpleCNN(num_channels=32).to(device)
student_model = SimpleCNN(num_channels=32).to(device)

print("Teacher params:", sum(p.numel() for p in teacher_model.parameters()))
print("Student params:", sum(p.numel() for p in student_model.parameters()))

Teacher params: 282250
Student params: 282250


La Loss de Distillation (Cœur du sujet)

In [8]:
def distillation_loss(student_logits, teacher_logits, labels, T=5.0, alpha=0.7):
    """
    student_logits: Sortie du réseau Student (avant Softmax)
    teacher_logits: Sortie du réseau Teacher (avant Softmax)
    labels: Vrais labels (Ground Truth)
    T: Température
    alpha: Poids de la perte de distillation (vs perte classique)
    """
    
    # 1. Perte "Soft" (Distillation)
    # Note: KLDivLoss attend log_softmax en entrée et softmax en cible
    distillation_loss = nn.KLDivLoss(reduction="batchmean")(
        F.log_softmax(student_logits / T, dim=1),
        F.softmax(teacher_logits / T, dim=1)
    ) * (T * T) # On multiplie par T^2 pour garder l'échelle des gradients
    
    # 2. Perte "Hard" (Classification classique sur les labels)
    student_loss = F.cross_entropy(student_logits, labels)
    
    # 3. Combinaison
    total_loss = alpha * distillation_loss + (1 - alpha) * student_loss
    
    return total_loss

Boucle d'entraînement (Comparaison)

In [9]:
# On gèle le teacher (il ne doit pas apprendre ici, il est censé savoir)
teacher_model.eval() 

optimizer = optim.SGD(student_model.parameters(), lr=LR, momentum=0.9)

print("Début de la distillation...")

for epoch in range(EPOCHS):
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        inputs, labels = data[0].to(device), data[1].to(device)

        # Zéro gradients
        optimizer.zero_grad()

        # Forward Student
        student_logits = student_model(inputs)

        # Forward Teacher (Sans calculer de gradients !)
        with torch.no_grad():
            teacher_logits = teacher_model(inputs)

        # Calcul de la Loss
        loss = distillation_loss(student_logits, teacher_logits, labels, T=4.0, alpha=0.5)

        # Backward & Optimize
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        
    print(f"Epoch {epoch+1}, Loss: {running_loss / len(trainloader):.4f}")

print("Distillation terminée.")

Début de la distillation...
Epoch 1, Loss: 1.0070
Epoch 2, Loss: 0.8914
Epoch 3, Loss: 0.8339
Distillation terminée.


**Partie Avancée : Comprendre DINO (Collpase Avoidance)**

Mise à jour EMA du Teacher (L'Élève forme le Maître)

In [10]:
class DINOTeacherUpdate:
    def __init__(self, teacher, student, momentum=0.996):
        self.teacher = teacher
        self.student = student
        self.momentum = momentum # lambda dans le cours

    @torch.no_grad()
    def update(self):
        # Pour chaque paramètre du Student, on met à jour celui du Teacher
        for param_s, param_t in zip(self.student.parameters(), self.teacher.parameters()):
            # theta_t = m * theta_t + (1 - m) * theta_s
            param_t.data.mul_(self.momentum).add_((1 - self.momentum) * param_s.data)

# Test rapide
updater = DINOTeacherUpdate(teacher_model, student_model)
updater.update()
print("Mise à jour EMA effectuée (les poids du Teacher ont légèrement bougé vers ceux du Student).")

Mise à jour EMA effectuée (les poids du Teacher ont légèrement bougé vers ceux du Student).


Centrage et Affinement (Centering & Sharpening)

In [11]:
def dino_mechanisms_demo():
    # Simulons des logits bruts pour un batch de 4 images et 5 classes
    # Imaginons que le modèle commence à vouloir "collapser" (tout prédire pareil)
    logits = torch.tensor([
        [2.0, 2.0, 2.0, 2.0, 2.0], # Image 1 : modèle indécis
        [2.1, 1.9, 2.0, 2.0, 2.0], # Image 2
        [2.0, 2.0, 2.0, 2.1, 1.9], # Image 3
        [2.0, 2.0, 2.0, 2.0, 2.0]  # Image 4
    ])
    
    print("--- 1. Logits Bruts (Risque d'uniformité) ---")
    print(F.softmax(logits, dim=1))
    
    # SHARPENING (Affinement) : Basse température
    # Cela force le modèle à faire un choix
    temp_sharp = 0.1
    sharpened = F.softmax(logits / temp_sharp, dim=1)
    print("\n--- 2. Après Sharpening (T=0.1) ---")
    print("Le modèle fait des choix plus tranchés :")
    print(sharpened)
    
    # CENTERING (Centrage)
    # Imaginons que le modèle ait un biais pour la classe 0
    center = torch.tensor([1.5, 0, 0, 0, 0]) # Biais moyen observé
    
    centered_logits = logits - center
    print("\n--- 3. Après Centering (Suppression du biais) ---")
    print(centered_logits)
    
    # Combiné (DINO output pour le Teacher)
    final_output = F.softmax(centered_logits / temp_sharp, dim=1)
    print("\n--- 4. Sortie Finale Teacher (Centrée + Affinée) ---")
    print(final_output)

dino_mechanisms_demo()

--- 1. Logits Bruts (Risque d'uniformité) ---
tensor([[0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.2206, 0.1806, 0.1996, 0.1996, 0.1996],
        [0.1996, 0.1996, 0.1996, 0.2206, 0.1806],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000]])

--- 2. Après Sharpening (T=0.1) ---
Le modèle fait des choix plus tranchés :
tensor([[0.2000, 0.2000, 0.2000, 0.2000, 0.2000],
        [0.4466, 0.0604, 0.1643, 0.1643, 0.1643],
        [0.1643, 0.1643, 0.1643, 0.4466, 0.0604],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000]])

--- 3. Après Centering (Suppression du biais) ---
tensor([[0.5000, 2.0000, 2.0000, 2.0000, 2.0000],
        [0.6000, 1.9000, 2.0000, 2.0000, 2.0000],
        [0.5000, 2.0000, 2.0000, 2.1000, 1.9000],
        [0.5000, 2.0000, 2.0000, 2.0000, 2.0000]])

--- 4. Sortie Finale Teacher (Centrée + Affinée) ---
tensor([[7.6476e-08, 2.5000e-01, 2.5000e-01, 2.5000e-01, 2.5000e-01],
        [2.4690e-07, 1.0923e-01, 2.9692e-01, 2.9692e-01, 2.9692e-01],
        [6.0144e-08, 1.9661e